# Code to preprocess institutional data

In [ ]:
import pandas as pd
import geopandas as gpd
from tqdm import tqdm  
import re

# import local modules
from andeangc import data_update, data_homogenize
from andeangc import config as cfg

## Peru data (SENAMHI)

In [ ]:
# Usage workflow:
file_paths = list(cfg.RESOURCES.glob('SENAMHI_PERU/raw/*.xlsx'))

# Step 1: Process all files (this is working as expected)
metadata, timeseries = data_homogenize.process_excel_files(file_paths)

# Step 2: Clean metadata and timeseries
timeseries_clean = timeseries.loc[:, ~timeseries.columns.duplicated(keep='first')]
timeseries_clean = timeseries_clean[timeseries_clean.index >= cfg.period_q[0]]
metadata_clean = metadata.drop_duplicates(subset=['gauge_name'])

# merge operator info (in a few cases there are many operators per station)
metadata_clean = metadata.groupby('gauge_name').agg({
    col: 'first' if col not in ['operator'] else lambda x: ', '.join(x.unique())
    for col in metadata.columns if col != 'gauge_name'})

metadata_clean = metadata_clean.sort_values(by='gauge_name').reset_index()

# gauge_id comes from the committed registry, never from the filename: every ANDREA
# export is called DatosSerie(N).xlsx, so N is a download counter, not a station code.
metadata_clean['gauge_id'] = data_homogenize.assign_gauge_ids(
    metadata_clean['gauge_name'],
    cfg.RESOURCES / 'gauge_ids_peru.csv',
    cfg.gauge_id_prefix_pe,
    cfg.gauge_id_zfill)
metadata_clean = metadata_clean.set_index('gauge_id')
timeseries_clean.columns = metadata_clean.index

# Step 3: remove stations with problems in gauge location
metadata_clean = metadata_clean[~metadata_clean.gauge_name.isin(cfg.senamhi_stations_to_remove)]
timeseries_clean = timeseries_clean[metadata_clean.index]

# Step 4: Save cleaned data and proceed with delineation
timeseries_clean.to_csv(cfg.RESOURCES / 'SENAMHI_PERU/SENAMHI_data.csv', index_label='date')
metadata_clean.to_csv(cfg.RESOURCES / 'SENAMHI_PERU/SENAMHI_metadata.csv')

# data missing in Chancos (Marcara), Colcas (Colcas), Balsa (Santa), Recreta (Santa), 
# Llanganuco (Llanganuco), Los Cedros (Los Cedros), Pachacoto (Pachacoto), Paron (Paron)
# Querococha (Querococha)

## Argentina data (SNHI)

In [ ]:
# Collect all Excel files (mean daily flow; in spanish caudal medio diario)
excel_files = list(cfg.RESOURCES.glob("SNHI_ARG/raw/*.xlsx"))

# Initialize an empty list to store dataframes
combined_df = []

# Loop through each file and append its data
for file in tqdm(excel_files):
    
    # Read the first row to extract the gauge information
    gauge_info = pd.read_excel(file, nrows=1).columns[0]
    
    # Extract the station number using regex
    match = re.search(r'Estacion (\d+)', gauge_info)
    if match:
        station_number = match.group(1)
        gauge_id = data_homogenize.format_gauge_ids(
            [station_number], cfg.gauge_id_prefix_arcl, cfg.gauge_id_zfill)[0]
    else:
        gauge_id = "UNKNOWN"  
    
    # Read the file, skipping the first row and using the second row as column names
    df = pd.read_excel(file, header=1)
    df = df.rename(columns = {"Caudal Medio Diario [m3/seg]": gauge_id})
    df['Fecha y Hora'] = pd.to_datetime(df['Fecha y Hora'], format='%d/%m/%Y %H:%M', errors='coerce').dt.date    
    df['Fecha y Hora'] = pd.to_datetime(df['Fecha y Hora'])
    df[gauge_id] = pd.to_numeric(df[gauge_id],  errors='coerce')
    df = df.set_index("Fecha y Hora")
    df = df[~df.index.duplicated(keep="first")]
    combined_df.append(df)

combined_df = pd.concat(combined_df, axis=1, sort=False)
combined_df.index.name = "date"
combined_df.index = pd.to_datetime(combined_df.index, errors="coerce")
combined_df = combined_df[combined_df.index.notna()]
combined_df = combined_df.sort_index()
combined_df = combined_df.loc[cfg.period_q[0]:cfg.period_q[1]]
combined_df.to_csv(cfg.RESOURCES / "SNHI_ARG/SNHI_data.csv")

## CAMELS-CL [Update]

In [ ]:
final_data = data_update.update_camels_cl_data(
    cfg.RESOURCES / "CAMELS_CL/CAMELS_CL_daily_1950_2020.csv",
    cfg.RESOURCES / "CAMELS_CL/DGA_1960_2025.parquet",
    cfg.RESOURCES / "CAMELS_CL/CAMELS_CL_daily_1950_2025.csv"
)

# CAMELS-CL is the only source whose metadata and basins still arrive keyed by the bare DGA
# station code (the series above are stamped inside update_camels_cl_data). Rewrite both in
# place with gauge_id first and the DGA code kept as gauge_id_source, so nb03 merges four
# sources that are all already keyed by gauge_id, and a second run rebuilds the same key.
camels_metadata = pd.read_csv(cfg.RESOURCES / "CAMELS_CL/CAMELS_CL_metadata.csv")
camels_metadata = data_homogenize.stamp_gauge_ids(
    camels_metadata, cfg.gauge_id_prefix_arcl, cfg.gauge_id_zfill)
camels_metadata.to_csv(cfg.RESOURCES / "CAMELS_CL/CAMELS_CL_metadata.csv", index=False)

camels_basins = gpd.read_file(cfg.RESOURCES / "CAMELS_CL/basins_CAMELS_CL.gpkg")
camels_basins = data_homogenize.stamp_gauge_ids(
    camels_basins, cfg.gauge_id_prefix_arcl, cfg.gauge_id_zfill)

# A GeoPackage cannot recreate a layer in place — writing over one either raises or leaves a
# second layer behind, and gpd.read_file would then hand nb03 the stale one. Write beside it
# and swap, naming the layer explicitly so it does not follow the temporary filename.
camels_basins_tmp = cfg.RESOURCES / "CAMELS_CL/basins_CAMELS_CL.tmp.gpkg"
camels_basins.to_file(camels_basins_tmp, layer="basins_CAMELS_CL")
camels_basins_tmp.replace(cfg.RESOURCES / "CAMELS_CL/basins_CAMELS_CL.gpkg")


## PMET-obs [Update]

In [ ]:
final_data = data_update.update_pmet_data(
    cfg.RESOURCES / "PMET_OBS/Q_PMETobs_1950_2020_v11d.csv",
    cfg.RESOURCES / "CAMELS_CL/DGA_1960_2025.parquet",
    cfg.RESOURCES / "SNHI_ARG/SNHI_data.csv",
    cfg.RESOURCES / "PMET_OBS/Q_PMETobs_1950_2025_v11d.csv"
)